# E2E Smoke Tests — FastAPI app

This notebook runs the FastAPI app in-process via `TestClient` with lightweight mocks for the LLM and KB layers so tests run quickly without Docker.

In [ ]:
import os
# Safe defaults for imports that read env at import time
os.environ.setdefault('OPENAI_API_KEY', 'test-openai-key')
os.environ.setdefault('DATABASE_URL', 'sqlite:///:memory:')
os.environ.setdefault('REDIS_URL', 'redis://localhost:6379/0')
print('Environment variables set')

In [ ]:
# Import app and patch external dependencies to deterministic test doubles
import main
from eval_service import EvalService
from kb_service import KBService

# Provide a no-op init_session_table to avoid DB-side table creation during tests
def noop_init_session_table(session_id: str):
    return f'kb_vectors_{session_id.replace('-', '_')}'

main.init_session_table = noop_init_session_table

# Patch EvalService to return deterministic responses and metrics
def fake_generate(self, model: str, messages: list) -> str:
    return f'Mocked response for {model}'

def fake_compute_metrics(self, output: str, context: str, ground_truth: str = '') -> dict:
    return {'faithfulness': 0.95, 'reason': 'mocked'}

EvalService.generate = fake_generate
EvalService.compute_metrics = fake_compute_metrics

# Patch KBService.retrieve to return simple nodes with .text attributes
class _Node:
    def __init__(self, text):
        self.text = text

def fake_retrieve(self, session_id: str, query: str, top_k: int = 3):
    return [_Node('Mocked context A'), _Node('Mocked context B')]

KBService.retrieve = fake_retrieve

print('Patched main.init_session_table, EvalService, and KBService')

In [ ]:
# Create TestClient and run a health check
from fastapi.testclient import TestClient
client = TestClient(main.app)
r = client.get('/health')
print('GET /health ->', r.status_code, r.json())

In [ ]:
# Create a session and run evaluate flow (using the patched services)
resp = client.post('/sessions')
print('POST /sessions ->', resp.status_code, resp.json())
session_id = resp.json().get('session_id')

body = {
    'query': 'Summarize the docs',
    'models': ['openai/gpt-4o-mini'],
    'ground_truth': 'Expected summary'
}

eval_resp = client.post(f'/sessions/{session_id}/evaluate', json=body)
print('POST /sessions/{session_id}/evaluate ->', eval_resp.status_code)
print(eval_resp.json())

# Basic assertions to fail the notebook if something breaks
assert eval_resp.status_code == 200, 'Evaluate endpoint failed'
data = eval_resp.json()
assert 'openai/gpt-4o-mini' in data, 'Missing model key in eval response'
print('E2E smoke test passed')

Done — the notebook ran a health check, created a session, and executed an evaluate request with mocked LLM/KB services.

To run non-mocked flows (ingestion, real LLMs), start the services via `docker compose up --build` and remove the patches in the notebook.